# Pair Trading with ECM-Guided Kalman Filter

This notebook keeps the original project data-cleaning and trading structure, but replaces the native random-walk Kalman filter with a train-only ECM-guided Kalman filter. All Engle-Granger/ECM parameters and Kalman noise parameters are fitted in sample, then frozen for OOS testing.

In [ ]:
import itertools
import math
import warnings
from datetime import date

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.vector_ar.vecm import coint_johansen

try:
    from pykalman import KalmanFilter
except ImportError:
    KalmanFilter = None
    print("pykalman is not installed. Run `%pip install pykalman` in a notebook cell before EM calibration.")

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
sns.set_style("whitegrid")


## Data Preprocessing

This section intentionally follows the original project data flow: load the three CSV files, clean raw ticks, choose the bar size from aligned coverage, then construct pair VWAP series.

In [ ]:
def get_data(): 
    A = pd.read_csv(r"data\A.csv")
    B = pd.read_csv(r"data\B.csv")
    C = pd.read_csv(r"data\C.csv")
    return A, B, C

def data_analysis(df, name="X"):
    print("Data Description:", "\n")
    print(df.describe())

    require = ["exg_time", "trade_price", "trade_qty"]
    missing = [c for c in require if c not in df.columns]
    if missing:
        return {"name": name, "error": f"missing columns: {missing}"}

    d = df.copy()
    d["exg_time"] = pd.to_datetime(d["exg_time"], utc=True, errors="coerce")
    d = d.sort_values("exg_time")

    out = {}
    out["name"] = name
    out["rows"] = len(d)
    out["null_exg_time"] = int(d["exg_time"].isna().sum())
    out["null_price"] = int(d["trade_price"].isna().sum())
    out["null_qty"] = int(d["trade_qty"].isna().sum())
    out["nonpos_price"] = int((d["trade_price"] <= 0).sum())
    out["nonpos_qty"] = int((d["trade_qty"] <= 0).sum())
    out["start"] = str(d["exg_time"].min())
    out["end"] = str(d["exg_time"].max())

    keys = ["exg_time", "trade_price", "trade_qty"]
    dup_mask = df.duplicated(subset=keys, keep="first")
    out["duplicated rows"] = int(dup_mask.sum())
    return out

def data_cleaning(df):
    tick_keys = ["exg_time", "trade_price", "trade_qty"]
    d = df.copy()
    d["exg_time"] = pd.to_datetime(d["exg_time"], utc=True, errors="coerce")

    d = d.dropna(subset=["exg_time"])
    print("null time drop count:", len(df) - len(d))

    d = d[(d["trade_price"] > 0) & (d["trade_qty"] > 0)]
    before_dupes = len(d)
    d = d.drop_duplicates(subset=tick_keys, keep="first")
    print("exact duplicate drop count:", before_dupes - len(d))

    d = d.sort_values("exg_time").reset_index(drop=True)
    return d

def make_vwap_bars(df, freq="15min", time_col="exg_time", price_col="trade_price", qty_col="trade_qty"):
    d = df.copy()
    d[time_col] = pd.to_datetime(d[time_col], utc=True, errors="coerce")
    d = d.dropna(subset=[time_col]).sort_values(time_col)
    d = d[(d[price_col] > 0) & (d[qty_col] > 0)]

    d["pxq"] = d[price_col] * d[qty_col]
    g = d.set_index(time_col).groupby(pd.Grouper(freq=freq))

    vol = g[qty_col].sum()
    vwap = g["pxq"].sum() / vol

    out = pd.DataFrame({"vwap": vwap, "vol": vol})
    out = out[(out["vol"] > 0) & out["vwap"].notna()]
    return out

def choose_x(df1_clean, df2_clean, candidates=(5, 15, 30), min_aligned=1500):
    """Choose the smallest candidate bar size with enough aligned pair observations."""
    t1 = pd.to_datetime(df1_clean["exg_time"], utc=True, errors="coerce").dropna()
    t2 = pd.to_datetime(df2_clean["exg_time"], utc=True, errors="coerce").dropna()
    start = max(t1.min(), t2.min())
    end = min(t1.max(), t2.max())

    best = None
    diag = {}
    for m in candidates:
        freq = f"{m}min"
        b1 = make_vwap_bars(df1_clean, freq=freq).loc[start:end, "vwap"]
        b2 = make_vwap_bars(df2_clean, freq=freq).loc[start:end, "vwap"]
        aligned = pd.concat([b1, b2], axis=1).dropna()
        print("alignment drop count:", max(len(b1), len(b2)) - len(aligned))
        diag[m] = {"aligned_bars": int(len(aligned))}
        if len(aligned) >= min_aligned and best is None:
            best = m

    if best is None:
        best = candidates[-1]
    return best, diag

def build_pair_series(df1_clean, df2_clean, x_minutes, name1="X1", name2="X2"):
    freq = f"{x_minutes}min"
    b1 = make_vwap_bars(df1_clean, freq=freq)["vwap"].rename(name1)
    b2 = make_vwap_bars(df2_clean, freq=freq)["vwap"].rename(name2)
    pair = pd.concat([b1, b2], axis=1).dropna()
    return pair[name1], pair[name2], pair

def cleaned_data():
    A, B, C = get_data()

    print(data_analysis(A, "A"))
    print(data_analysis(B, "B"))
    print(data_analysis(C, "C"))

    A_clean = data_cleaning(A)
    B_clean = data_cleaning(B)
    C_clean = data_cleaning(C)

    x_AB, diag_AB = choose_x(A_clean, B_clean)
    x_AC, diag_AC = choose_x(A_clean, C_clean)
    x_BC, diag_BC = choose_x(B_clean, C_clean)

    _, _, AB = build_pair_series(A_clean, B_clean, x_AB, "A", "B")
    _, _, AC = build_pair_series(A_clean, C_clean, x_AC, "A", "C")
    _, _, BC = build_pair_series(B_clean, C_clean, x_BC, "B", "C")

    return {
        "A_clean": A_clean, "B_clean": B_clean, "C_clean": C_clean,
        "x_AB": x_AB, "x_AC": x_AC, "x_BC": x_BC,
        "diag_AB": diag_AB, "diag_AC": diag_AC, "diag_BC": diag_BC,
        "AB": AB, "AC": AC, "BC": BC,
    }


In [ ]:
# Build cleaned pair data once.
res = cleaned_data()
print("best x min for AB:", res["x_AB"])
print("best x min for AC:", res["x_AC"])
print("best x min for BC:", res["x_BC"])

## ECM-Guided Kalman Filter

The hidden state is `[beta_t, alpha_t]`. The prior for each bar is driven by the ECM adjustment speed estimated on the train sample: `beta_prior_t = (1 + lambda) * beta_post_{t-1}` and `alpha_prior_t = alpha_post_{t-1}`. `pykalman` is used only for train-sample EM calibration of `Q` and `R`; the actual filter recursion below is manual.

In [ ]:
def fit_eg_ecm(dep: pd.Series, ind: pd.Series, autolag="AIC") -> dict:
    """
    Train-only Engle-Granger + ECM fit for one ordering:
        dep_t = alpha + beta * ind_t + eps_t
        d(dep_t) = c + lambda * eps_{t-1} + gamma * d(dep_{t-1}) + delta * d(ind_{t-1}) + u_t
    """
    dep = dep.astype(float).dropna()
    ind = ind.astype(float).reindex(dep.index).dropna()
    dep = dep.reindex(ind.index)

    lr = sm.OLS(dep, sm.add_constant(ind)).fit()
    alpha = float(lr.params.iloc[0])
    beta = float(lr.params.iloc[1])
    resid = lr.resid

    adf_stat, adf_p, *_ = adfuller(resid.dropna(), autolag=autolag)

    ecm_df = pd.DataFrame({
        "dy": dep.diff(),
        "ect_lag": resid.shift(1),
        "dy_lag": dep.diff().shift(1),
        "dx_lag": ind.diff().shift(1),
    }).dropna()

    ecm = sm.OLS(ecm_df["dy"], sm.add_constant(ecm_df[["ect_lag", "dy_lag", "dx_lag"]])).fit()
    lam = float(ecm.params["ect_lag"])
    lam_p = float(ecm.pvalues["ect_lag"])

    return {
        "alpha": alpha,
        "beta": beta,
        "r_squared": float(lr.rsquared),
        "adf_stat": float(adf_stat),
        "adf_p": float(adf_p),
        "lambda_raw": lam,
        "lambda_p": lam_p,
    }

def effective_ecm_lambda(lam: float, lower_bound: float = -0.99) -> float:
    """
    Positive ECM coefficients are not error-correcting. For OOS filtering, use the
    mean-reverting component only and cap extreme negative values for stability.
    """
    if not np.isfinite(lam):
        return 0.0
    return float(np.clip(min(lam, 0.0), lower_bound, 0.0))

def half_life_from_lambda(lam: float) -> float:
    if lam < 0 and 0 < (1.0 + lam) < 1.0:
        return float(np.log(0.5) / np.log(1.0 + lam))
    return np.nan

def build_obs_matrix(ind_vals: np.ndarray) -> np.ndarray:
    H = np.zeros((len(ind_vals), 1, 2), dtype=float)
    H[:, 0, 0] = ind_vals
    H[:, 0, 1] = 1.0
    return H

def calibrate_qr_em(dep_train: pd.Series, ind_train: pd.Series, alpha0: float, beta0: float, lam: float, n_iter: int = 20):
    """Use pykalman only for train-sample EM calibration of transition Q and observation R."""
    if KalmanFilter is None:
        raise ImportError("pykalman is required for EM calibration. Run `%pip install pykalman` and restart this notebook kernel.")

    y = dep_train.astype(float).values.reshape(-1, 1)
    x = ind_train.astype(float).values
    H = build_obs_matrix(x)
    F = np.array([[1.0 + lam, 0.0], [0.0, 1.0]], dtype=float)

    kf = KalmanFilter(
        n_dim_obs=1,
        n_dim_state=2,
        transition_matrices=F,
        observation_matrices=H,
        initial_state_mean=np.array([beta0, alpha0], dtype=float),
        initial_state_covariance=np.eye(2),
        em_vars=["transition_covariance", "observation_covariance"],
    )
    fitted = kf.em(y, n_iter=n_iter)

    Q = np.asarray(fitted.transition_covariance, dtype=float)
    Q = (Q + Q.T) / 2.0
    Q += np.eye(2) * 1e-12
    R = float(np.asarray(fitted.observation_covariance).reshape(-1)[0])
    R = max(R, 1e-12)
    return Q, R

def manual_ecm_kalman_filter(ind: pd.Series, dep: pd.Series, alpha0: float, beta0: float, lam: float, Q: np.ndarray, R: float, P0=None):
    """
    Manual ECM-KF recursion. The prior is explicit:
        [beta_prior, alpha_prior] = F @ [beta_post_lag, alpha_post_lag]
    where F = [[1 + lambda, 0], [0, 1]].
    """
    x = ind.astype(float)
    y = dep.astype(float).reindex(x.index)
    if not x.index.equals(y.index):
        raise ValueError("ind and dep must share the same index.")

    F = np.array([[1.0 + lam, 0.0], [0.0, 1.0]], dtype=float)
    Q = np.asarray(Q, dtype=float)
    R = float(R)
    I = np.eye(2)

    state = np.array([beta0, alpha0], dtype=float)
    P = np.eye(2) if P0 is None else np.asarray(P0, dtype=float)

    rows = []
    for ts, xt, yt in zip(x.index, x.values, y.values):
        state_prior = F @ state
        P_prior = F @ P @ F.T + Q
        H = np.array([[xt, 1.0]], dtype=float)
        innovation = float(yt - (H @ state_prior)[0])
        S = float((H @ P_prior @ H.T)[0, 0] + R)
        K = (P_prior @ H.T) / S
        state_post = state_prior + (K[:, 0] * innovation)
        P_post = (I - K @ H) @ P_prior

        beta_post, alpha_post = state_post
        rows.append({
            "date": ts,
            "beta_prior": state_prior[0],
            "alpha_prior": state_prior[1],
            "beta_hat": beta_post,
            "alpha_hat": alpha_post,
            "P_beta": P_post[0, 0],
            "P_alpha": P_post[1, 1],
            "innovation": innovation,
            "innovation_var": S,
            "kalman_gain_beta": K[0, 0],
            "kalman_gain_alpha": K[1, 0],
            "spread": float(yt - (alpha_post + beta_post * xt)),
        })

        state = state_post
        P = P_post

    return pd.DataFrame(rows).set_index("date")


## Trading and Analytics

Signals are generated from the ECM-KF residual spread. Z-score statistics are rolling and shifted by one bar, and trading uses lagged z-score and lagged beta so no current-bar signal is traded on the same observation.

In [ ]:
def cadf_test(df, x, y, add_const=True, autolag="AIC"):
    X = df[x].astype(float).values
    Y = df[y].astype(float).values

    if add_const:
        Xreg = sm.add_constant(X)
        res = sm.OLS(Y, Xreg).fit()
        alpha, beta = res.params
        resid = Y - (alpha + beta * X)
    else:
        res = sm.OLS(Y, X).fit()
        beta = res.params[0]
        alpha = 0.0
        resid = Y - beta * X

    adf_res = adfuller(resid, autolag=autolag, regression="c")
    return {"adf_stat": float(adf_res[0]), "p_value": float(adf_res[1])}

def johansen_trace_test(df, x_col, y_col):
    pair = df[[x_col, y_col]].dropna().astype(float)
    j = coint_johansen(pair, det_order=0, k_ar_diff=1)
    trace_r0 = float(j.lr1[0])
    crit95_r0 = float(j.cvt[0, 1])
    return {"trace_r0": trace_r0, "crit95_r0": crit95_r0, "reject_r0_95": trace_r0 > crit95_r0}

def plot_spread_acf(spread: pd.Series, lags: int = 60, title: str = "ACF of spread"):
    s = pd.Series(spread).dropna().astype(float)
    plt.figure(figsize=(9, 4))
    plot_acf(s.values, lags=lags, zero=False)
    plt.title(title)
    plt.xlabel("Lag")
    plt.ylabel("Autocorrelation")
    plt.tight_layout()
    plt.show()

class TradingEngine:
    def __init__(self, initial_capital=100000.0, commission_bps=0):
        self.cash = initial_capital
        self.initial_capital = initial_capital
        self.commission_bps = commission_bps
        self.positions = {}
        self.history = []
        self.equity_curve = []

    def get_portfolio_value(self, current_prices):
        market_value = 0.0
        for ticker, pos in self.positions.items():
            if ticker in current_prices:
                market_value += pos["qty"] * current_prices[ticker]
        return self.cash + market_value

    def execute_order(self, ticker, qty, price, date, action_label):
        if qty == 0 or pd.isna(qty) or pd.isna(price):
            return

        notional = abs(qty * price)
        comm = notional * self.commission_bps
        self.cash -= qty * price
        self.cash -= comm

        current_pos = self.positions.get(ticker, {"qty": 0.0, "avg_price": 0.0})
        curr_qty = current_pos["qty"]
        new_total_qty = curr_qty + qty

        if curr_qty == 0 or (curr_qty * qty) > 0:
            old_notional = curr_qty * current_pos["avg_price"]
            new_trade_notional = qty * price
            new_avg_price = price if new_total_qty == 0 else (old_notional + new_trade_notional) / new_total_qty
            self.positions[ticker] = {"qty": new_total_qty, "avg_price": new_avg_price}
        else:
            if abs(new_total_qty) < 1e-10:
                self.positions.pop(ticker, None)
            else:
                is_flip = (curr_qty * new_total_qty) < 0
                new_avg_price = price if is_flip else current_pos["avg_price"]
                self.positions[ticker] = {"qty": new_total_qty, "avg_price": new_avg_price}

        self.history.append({"date": date, "ticker": ticker, "action": action_label, "qty": qty, "price": price, "comm": comm})

def portfolio_analytics(equity: pd.Series):
    equity = equity.dropna().astype(float)
    if equity.empty:
        return {}, pd.Series(dtype=float), pd.Series(dtype=float)

    equity_daily = equity.resample("B").last().dropna()
    rets_daily = equity_daily.pct_change().dropna()

    sharpe = 0.0 if rets_daily.std() == 0 or rets_daily.empty else (rets_daily.mean() / rets_daily.std()) * np.sqrt(252)
    ann_vol = 0.0 if rets_daily.empty else rets_daily.std() * np.sqrt(252)

    years = (equity_daily.index[-1] - equity_daily.index[0]).days / 365.25 if len(equity_daily) > 1 else 0.0
    cagr = (equity_daily.iloc[-1] / equity_daily.iloc[0]) ** (1 / years) - 1 if years > 0 else 0.0

    peak = equity.cummax()
    dd = equity / peak - 1.0
    max_dd = dd.min()

    peak_daily = equity_daily.cummax()
    underwater_daily = equity_daily < peak_daily
    dd_duration = (underwater_daily.groupby((underwater_daily != underwater_daily.shift()).cumsum()).cumcount() + 1)
    max_dd_duration = dd_duration[underwater_daily].max() if underwater_daily.any() else 0

    return {
        "Final Equity": float(equity.iloc[-1]),
        "Total Return": float(equity.iloc[-1] / equity.iloc[0] - 1),
        "CAGR": float(cagr),
        "Ann Vol": float(ann_vol),
        "Sharpe (rf=0)": float(sharpe),
        "Max Drawdown": float(max_dd),
        "Max DD Duration (days)": int(max_dd_duration),
    }, dd, rets_daily


In [ ]:
def run_z_score_backtest_ecm(
    res,
    engine,
    X="A",
    Y="B",
    pair_name="AB",
    entry_z=2.0,
    exit_z=0.0,
    window=20,
    split=(0.6, 0.4),
    em_iter=20,
    commission_bps=0,
    verbose=True,
):
    """
    No-look-ahead ECM-KF backtest.
    All ECM and Q/R parameters are fitted on train only. Trading metrics are OOS only.
    """
    if pair_name not in res:
        raise ValueError(f"Pair {pair_name} not found in results dictionary.")

    df = res[pair_name].dropna().copy()
    n = len(df)
    n_train = int(n * split[0])
    if n_train <= max(window, 20) or n_train >= n:
        raise ValueError("split leaves too little train or OOS data.")

    train = df.iloc[:n_train].copy()
    oos_start_idx = n_train
    train_end = train.index[-1]
    oos_start = df.index[oos_start_idx]

    x_train = train[X].astype(float)
    y_train = train[Y].astype(float)

    ecm = fit_eg_ecm(y_train, x_train)
    alpha0 = ecm["alpha"]
    beta0 = ecm["beta"]
    lam_raw = ecm["lambda_raw"]
    lam_eff = effective_ecm_lambda(lam_raw)
    half_life = half_life_from_lambda(lam_eff)

    Q, R = calibrate_qr_em(y_train, x_train, alpha0=alpha0, beta0=beta0, lam=lam_eff, n_iter=em_iter)

    kf_df = manual_ecm_kalman_filter(
        ind=df[X],
        dep=df[Y],
        alpha0=alpha0,
        beta0=beta0,
        lam=lam_eff,
        Q=Q,
        R=R,
        P0=np.eye(2),
    )

    model_df = df.join(kf_df, how="left")
    roll_mean = model_df["spread"].rolling(window=window, min_periods=window).mean().shift(1)
    roll_std = model_df["spread"].rolling(window=window, min_periods=window).std().shift(1)
    model_df["z_score"] = (model_df["spread"] - roll_mean) / roll_std
    model_df["z_trade"] = model_df["z_score"].shift(1)
    model_df["beta_trade"] = model_df["beta_hat"].shift(1)
    model_df["is_oos"] = False
    model_df.iloc[oos_start_idx:, model_df.columns.get_loc("is_oos")] = True

    if verbose:
        cadf_res = cadf_test(train, X, Y)
        johansen_res = johansen_trace_test(train, X, Y)
        print(f"\n--- ECM-KF OOS Backtest: {Y}_vs_{X} ({pair_name}) ---")
        print(f"Train rows: {len(train):,} | OOS rows: {len(df) - len(train):,}")
        print(f"Train end: {train_end} | OOS start: {oos_start}")
        print(f"Train CADF p-value: {cadf_res['p_value']:.5f} | Johansen reject r=0 @95%: {johansen_res['reject_r0_95']}")
        print(f"alpha0: {alpha0:.6f} | beta0: {beta0:.6f}")
        print(f"lambda raw: {lam_raw:.6f} | lambda effective: {lam_eff:.6f} | lambda p: {ecm['lambda_p']:.5f}")
        print(f"half-life: {half_life:.2f} bars" if np.isfinite(half_life) else "half-life: N/A")
        print(f"Q diag: {np.diag(Q)} | R: {R:.6e}")
        print("Starting OOS trading simulation...")

    current_state = 0
    base_qty = 100.0
    for index, row in model_df.iloc[oos_start_idx:].iterrows():
        price_x = row[X]
        price_y = row[Y]
        z = row["z_trade"]
        beta = row["beta_trade"]

        if not (pd.isna(z) or pd.isna(beta)):
            hedge_qty = base_qty * beta
            if current_state == 0 and z < -entry_z:
                engine.execute_order(Y, base_qty, price_y, index, "BUY Y (Long Spread)")
                engine.execute_order(X, -hedge_qty, price_x, index, "SELL X (Hedge)")
                current_state = 1
            elif current_state == 0 and z > entry_z:
                engine.execute_order(Y, -base_qty, price_y, index, "SELL Y (Short Spread)")
                engine.execute_order(X, hedge_qty, price_x, index, "BUY X (Hedge)")
                current_state = -1
            elif current_state == 1 and z >= -exit_z:
                if Y in engine.positions:
                    engine.execute_order(Y, -engine.positions[Y]["qty"], price_y, index, "CLOSE Y")
                if X in engine.positions:
                    engine.execute_order(X, -engine.positions[X]["qty"], price_x, index, "CLOSE X")
                current_state = 0
            elif current_state == -1 and z <= exit_z:
                if Y in engine.positions:
                    engine.execute_order(Y, -engine.positions[Y]["qty"], price_y, index, "CLOSE Y")
                if X in engine.positions:
                    engine.execute_order(X, -engine.positions[X]["qty"], price_x, index, "CLOSE X")
                current_state = 0

        prices = {X: price_x, Y: price_y}
        engine.equity_curve.append({"date": index, "equity": engine.get_portfolio_value(prices)})

    equity = pd.DataFrame(engine.equity_curve).set_index("date")
    fitted_params = {
        "pair_name": pair_name,
        "X": X,
        "Y": Y,
        "train_end": train_end,
        "oos_start": oos_start,
        "n_train": len(train),
        "n_oos": len(df) - len(train),
        "alpha0": alpha0,
        "beta0": beta0,
        "lambda_raw": lam_raw,
        "lambda_effective": lam_eff,
        "lambda_p": ecm["lambda_p"],
        "half_life_bars": half_life,
        "Q": Q,
        "R": R,
        "window": window,
        "entry_z": entry_z,
        "exit_z": exit_z,
        "em_iter": em_iter,
        "split": split,
    }

    if verbose:
        print("OOS simulation complete.")
    return equity, fitted_params, model_df

def run_all_permutations_ecm(res, split=(0.6, 0.4), entry_z=2.0, exit_z=0.0, window=20, em_iter=20, commission_bps=0):
    results = {}
    stats_dict = {}
    fitted_params_dict = {}
    model_frames = {}

    permutations = [
        ("AB", "A", "B"),
        ("AB", "B", "A"),
        ("AC", "A", "C"),
        ("AC", "C", "A"),
        ("BC", "B", "C"),
        ("BC", "C", "B"),
    ]

    for key, x_col, y_col in permutations:
        strat_name = f"{y_col}_vs_{x_col}"
        engine = TradingEngine(initial_capital=100000.0, commission_bps=commission_bps)
        try:
            equity, fitted_params, model_df = run_z_score_backtest_ecm(
                res=res,
                engine=engine,
                X=x_col,
                Y=y_col,
                pair_name=key,
                entry_z=entry_z,
                exit_z=exit_z,
                window=window,
                split=split,
                em_iter=em_iter,
                commission_bps=commission_bps,
                verbose=True,
            )
            results[strat_name] = equity
            fitted_params_dict[strat_name] = fitted_params
            model_frames[strat_name] = model_df
            stats, dd, rets = portfolio_analytics(equity["equity"])
            stats_dict[strat_name] = stats
            print(f"Portfolio Stats for {strat_name}:\n", pd.Series(stats))
        except Exception as exc:
            print(f"!! Error running {strat_name}: {exc}")

    return results, stats_dict, fitted_params_dict, model_frames


## Run OOS Backtests

All six pair directions are evaluated. The train split is used for ECM and EM calibration only. Portfolio statistics are computed from the OOS equity curve.

In [ ]:
all_strategies, stat_dict, fitted_params_dict, model_frames = run_all_permutations_ecm(
    res,
    split=(0.6, 0.4),
    entry_z=2.0,
    exit_z=0.0,
    window=20,
    em_iter=20,
    commission_bps=0,
)

## Results

In [ ]:
def plot_strategy_performance(results):
    plt.figure(figsize=(12, 7))
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(results), 1)))

    for (name, df), color in zip(results.items(), colors):
        if df.empty:
            continue
        start_equity = df["equity"].iloc[0]
        final_equity = df["equity"].iloc[-1]
        total_ret = ((final_equity / start_equity) - 1) * 100
        plt.plot(df.index, df["equity"], label=f"{name} ({total_ret:+.1f}%)", linewidth=2, alpha=0.85, color=color)

    plt.title("ECM-KF Pairs Trading: OOS Strategy Comparison", fontsize=14, pad=15)
    plt.ylabel("Portfolio Equity ($)", fontsize=12)
    plt.xlabel("Date", fontsize=12)
    plt.legend(loc="best", fontsize=10)
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

def summarize_fitted_params(fitted_params_dict):
    rows = []
    for name, p in fitted_params_dict.items():
        rows.append({
            "strategy": name,
            "pair": p["pair_name"],
            "Y": p["Y"],
            "X": p["X"],
            "train_end": p["train_end"],
            "oos_start": p["oos_start"],
            "alpha0": p["alpha0"],
            "beta0": p["beta0"],
            "lambda_raw": p["lambda_raw"],
            "lambda_effective": p["lambda_effective"],
            "lambda_p": p["lambda_p"],
            "half_life_bars": p["half_life_bars"],
            "R": p["R"],
            "Q_beta": p["Q"][0, 0],
            "Q_alpha": p["Q"][1, 1],
        })
    return pd.DataFrame(rows).set_index("strategy") if rows else pd.DataFrame()

def summarize_stats(stats_dict):
    return pd.DataFrame(stats_dict).T if stats_dict else pd.DataFrame()

def plot_beta_alpha_spread(model_frames, fitted_params_dict, strategy_name):
    model_df = model_frames[strategy_name]
    params = fitted_params_dict[strategy_name]
    oos_start = params["oos_start"]

    fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
    model_df["beta_hat"].plot(ax=axes[0], label="beta_hat", color="#2271B2")
    axes[0].axvline(oos_start, color="red", linestyle="--", alpha=0.6, label="OOS start")
    axes[0].set_title(f"{strategy_name}: ECM-KF beta")
    axes[0].legend()

    model_df["alpha_hat"].plot(ax=axes[1], label="alpha_hat", color="#2A9D8F")
    axes[1].axvline(oos_start, color="red", linestyle="--", alpha=0.6)
    axes[1].set_title(f"{strategy_name}: ECM-KF intercept")
    axes[1].legend()

    model_df["z_score"].plot(ax=axes[2], label="z_score", color="#E76F51")
    axes[2].axhline(2.0, color="black", linewidth=0.8, linestyle=":")
    axes[2].axhline(-2.0, color="black", linewidth=0.8, linestyle=":")
    axes[2].axvline(oos_start, color="red", linestyle="--", alpha=0.6)
    axes[2].set_title(f"{strategy_name}: spread z-score")
    axes[2].legend()

    plt.tight_layout()
    plt.show()


In [ ]:
params_summary = summarize_fitted_params(fitted_params_dict)
stats_summary = summarize_stats(stat_dict)

display(params_summary)
display(stats_summary)
plot_strategy_performance(all_strategies)

In [ ]:
# Optional: inspect one strategy's beta, intercept, and z-score path.
# Change the key to any value in fitted_params_dict.keys().
if fitted_params_dict:
    example_strategy = next(iter(fitted_params_dict))
    print("Example strategy:", example_strategy)
    plot_beta_alpha_spread(model_frames, fitted_params_dict, example_strategy)

## No-Look-Ahead Checklist

- ECM `alpha`, `beta`, `lambda`, and `lambda_p` are estimated on the train slice only.
- `Q` and `R` are EM-calibrated on the train slice only.
- OOS filtering uses frozen train-fitted parameters.
- The OOS state path naturally starts from the final train posterior because the manual filter is recursive.
- Z-score and beta used for trading are shifted by one bar.
- Reported performance uses only OOS equity curves.